In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import cholesky
import os
import sys

sys.path.append(os.path.abspath(".."))  # Adds the parent directory to sys.path

from privateggm.model import optimize_lbfgs, mirror_descent
from privateggm.utils import *
from privateggm.data import *
from privateggm.privacy import *
from privateggm.driver import *

# MaxEnt and Mirror Descent test (functions in model.py)

In [2]:
# generate a toy 3x3 covariance example

def generate_observed_indices(d, rng, perc_offdiag=.5):
    """select set E of observed indices"""

    # diagonal indices
    diag_inds = [(i, i) for i in range(d)]

    # all lower-tri off-diagonal indices (i > j)
    offdiag_inds = [(i, j) for i in range(1, d) for j in range(i)]
    n_total_offdiag = len(offdiag_inds)

    # num of off-diagonal entries to sample
    n_sample = int(np.round(perc_offdiag * n_total_offdiag))

    # randomly select subset
    selected_inds = rng.choice(n_total_offdiag, size=n_sample, replace=False)
    sampled_offdiags = [offdiag_inds[i] for i in selected_inds]

    # Combine diag and selected off-diags
    all_inds = diag_inds + sampled_offdiags

    return np.array(all_inds)

def generate_base_example(d, rng):
    """generate well behaved toy case"""

    A = rng.normal(loc=0.0, scale=1.0, size=(d, d))
    Sigma_true = A @ A.T + d * np.eye(d)

    return Sigma_true

def test_lbfgs(Sigma_tilde, inds, max_iters=1000, verbose=False):
    """
    Run the L-BFGS optimization on the toy example.
    """
    rows, cols = inds[:, 0], inds[:, 1]
    Sigma_tilde_vec = Sigma_tilde[rows, cols]
    z_init = initialize_z(Sigma_tilde, inds)
    
    d = Sigma_true.shape[0]  
    
    # Run lbfgs optimization
    z_opt, g_vals, obj_vals = optimize_lbfgs(Sigma_tilde_vec, z_init, inds, d, max_iters=max_iters, verbose=verbose)
    
    # Convert optimized z into a precision matrix K
    K_z_opt = get_K(z_opt, inds, d)
    
    # Convert back to covariance matrix
    Sigma_opt = np.linalg.inv(K_z_opt)

    return Sigma_opt

def test_mirror_descent(Sigma_tilde, inds, tau, max_iters=1000, eta_init=0.2, tol=1e-8, verbose=True):

    # Generate toy dataset
    Sigma_init = np.eye(Sigma_tilde.shape[0])

    d = Sigma_tilde.shape[0]
    
    # Run Mirror Descent
    Sigma_opt, K_opt = mirror_descent(Sigma_init, Sigma_tilde, inds, tau, 
                                      max_iters=max_iters, eta_init=eta_init, tol=tol, verbose=verbose)

    return Sigma_opt


def evaluate_covariance_estimate(Sigma_true, Sigma_tilde, Sigma_opt, inds):

    d = Sigma_true.shape[0]

    # Create boolean mask from indices
    mask = np.zeros((d, d), dtype=bool)
    for i, j in inds:
        mask[i, j] = True
        mask[j, i] = True  # Ensure symmetry

    # Compute errors
    mse_total = np.mean((Sigma_true - Sigma_opt)**2)
    obs_mse = np.mean((Sigma_tilde[mask] - Sigma_opt[mask])**2)
    true_obs_mse = np.mean((Sigma_true[mask] - Sigma_opt[mask])**2)

    # Print results
    print("\n" + "-"*50)
    print(f"True Covariance:\n{Sigma_true}")
    print(f"\nNoisy Covariance:\n{Sigma_tilde}")
    print(f"\nEstimated Covariance:\n{Sigma_opt}")
    print("\n" + "-"*50)
    print(f"MSE vs True (Total): {mse_total:.6f}")
    print(f"MSE vs True (Observed): {true_obs_mse:.6f}")
    print(f"MSE vs Noisy (Observed): {obs_mse:.6f}")
    print("-"*50 + "\n")

## 3 x 3 well-conditioned case

In [8]:
rng = np.random.default_rng(42)

# Define a well-conditioned covariance matrix
d = 3
Sigma_true = generate_base_example(d, rng)
# Sigma_true = np.array([[2.0, 0.5, 0.3],
#                            [0.5, 1.5, 0.4],
#                            [0.3, 0.4, 1.0]])
print("Sigma true", Sigma_true)

# Select set of observed indices E
inds = generate_observed_indices(d, rng, perc_offdiag=.67)

# Mask Sigma and add noise
noise_scale = 1.0
lower_triangle_noise = np.tril(rng.normal(0, noise_scale, Sigma_true.shape))

# tau in case no noise
if noise_scale == 0.0:
    tau = np.zeros_like(Sigma_true)
    tau[np.array(inds)[:,0], np.array(inds)[:,1]] = 1.0  # Set observed τ=1
else:
    tau = lower_triangle_noise + lower_triangle_noise.T - np.diag(np.diag(lower_triangle_noise))
    
Sigma_tilde = Sigma_true + tau

# Create mask from inds
mask = np.zeros_like(Sigma_true, dtype=bool)
for i, j in inds:
    mask[i, j] = True
    mask[j, i] = True  # ensure symmetry

# Zero out unobserved entries
Sigma_tilde = np.where(mask, Sigma_tilde, 0.0)
print(Sigma_tilde)

print("#" * 100)
print("\n******** LBFGS TEST ********\n")
print("#" * 100)

Sigma_opt_lbfgs = test_lbfgs(Sigma_tilde, inds, max_iters=2000, verbose=True)
evaluate_covariance_estimate(Sigma_true, Sigma_tilde, Sigma_opt_lbfgs, inds)

print("#" * 100)
print("\n******** MIRROR DESCENT TEST ********\n")
print("#" * 100)

Sigma_opt_md = test_mirror_descent(Sigma_tilde, inds, tau, max_iters=10000, eta_init=0.002, tol=1e-6, verbose=True)
evaluate_covariance_estimate(Sigma_true, Sigma_tilde, Sigma_opt_md, inds)

print("#" * 100)
print("\n******** PGD-MAXENT TEST ********\n")
print("#" * 100)

Sigma_pgd_maxent = pgd_maxent(Sigma_tilde, inds, max_iters=1000, eta=0.01, tol=1e-6, verbose=True)
evaluate_covariance_estimate(Sigma_true, Sigma_tilde, Sigma_pgd_maxent, inds)

Sigma true [[4.73759644 1.33842955 0.35523398]
 [1.33842955 9.38687176 0.75912072]
 [0.35523398 0.75912072 3.11663482]]
[[ 5.51538837  0.         -0.60364863]
 [ 0.          8.5275793   1.63757102]
 [-0.60364863  1.63757102  3.06670891]]
####################################################################################################

******** LBFGS TEST ********

####################################################################################################
Iter     0 | Obj: 7.971e+00 | Grad Sum: 4.482e+00 | η: 1.000e+00
Converged at iter     9 | ΔLoss: 2.259e-11

--------------------------------------------------
True Covariance:
[[4.73759644 1.33842955 0.35523398]
 [1.33842955 9.38687176 0.75912072]
 [0.35523398 0.75912072 3.11663482]]

Noisy Covariance:
[[ 5.51538837  0.         -0.60364863]
 [ 0.          8.5275793   1.63757102]
 [-0.60364863  1.63757102  3.06670891]]

Estimated Covariance:
[[ 5.51538836 -0.32233825 -0.60364866]
 [-0.32233825  8.52757906  1.6375711 ]
 [-0.

# 5x5 matrix

In [9]:
rng = np.random.default_rng(42)

# Define a well-conditioned covariance matrix
d = 5
Sigma_true = generate_base_example(d, rng)

print("Sigma true", Sigma_true)

# Select set of observed indices E
inds = generate_observed_indices(d, rng, perc_offdiag=.67)

# Mask Sigma and add noise
noise_scale = 1.0
lower_triangle_noise = np.tril(rng.normal(0, noise_scale, Sigma_true.shape))

# tau in case no noise
if noise_scale == 0.0:
    tau = np.zeros_like(Sigma_true)
    tau[np.array(inds)[:,0], np.array(inds)[:,1]] = 1.0  # Set observed τ=1
else:
    tau = lower_triangle_noise + lower_triangle_noise.T - np.diag(np.diag(lower_triangle_noise))
    
Sigma_tilde = Sigma_true + tau

# Create mask from inds
mask = np.zeros_like(Sigma_true, dtype=bool)
for i, j in inds:
    mask[i, j] = True
    mask[j, i] = True  # ensure symmetry

# Zero out unobserved entries
Sigma_tilde = np.where(mask, Sigma_tilde, 0.0)
print(Sigma_tilde)

print("#" * 100)
print("\n******** LBFGS TEST ********\n")
print("#" * 100)

Sigma_opt_lbfgs = test_lbfgs(Sigma_tilde, inds, max_iters=2000, verbose=True)
evaluate_covariance_estimate(Sigma_true, Sigma_tilde, Sigma_opt_lbfgs, inds)

print("#" * 100)
print("\n******** MIRROR DESCENT TEST ********\n")
print("#" * 100)

Sigma_opt_md = test_mirror_descent(Sigma_tilde, inds, tau, max_iters=10000, eta_init=0.002, tol=1e-6, verbose=True)
evaluate_covariance_estimate(Sigma_true, Sigma_tilde, Sigma_opt_md, inds)

print("#" * 100)
print("\n******** PGD-MAXENT TEST ********\n")
print("#" * 100)

Sigma_pgd_maxent = pgd_maxent(Sigma_tilde, inds, max_iters=1000, eta=0.01, tol=1e-6, verbose=True)
evaluate_covariance_estimate(Sigma_true, Sigma_tilde, Sigma_pgd_maxent, inds)

Sigma true [[11.42879673  0.88144319 -0.34325472 -0.44128407  2.25962047]
 [ 0.88144319  7.53999024 -1.48432747  1.49716382  0.13503275]
 [-0.34325472 -1.48432747  7.87189887  0.4347202  -0.9859031 ]
 [-0.44128407  1.49716382  0.4347202   7.56798405 -1.3788779 ]
 [ 2.25962047  0.13503275 -0.9859031  -1.3788779   7.19979055]]
[[10.916554    0.04128671  0.         -0.21768852  0.80246465]
 [ 0.04128671  6.71550902 -1.25216615  0.         -0.18463846]
 [ 0.         -1.25216615  7.98858468  0.50229926 -1.45627575]
 [-0.21768852  0.          0.50229926  7.85710345  0.        ]
 [ 0.80246465 -0.18463846 -1.45627575  0.          6.9246483 ]]
####################################################################################################

******** LBFGS TEST ********

####################################################################################################
Iter     0 | Obj: 1.537e+01 | Grad Sum: 8.914e+00 | η: 1.000e+00
Converged at iter     8 | ΔLoss: 7.696e-09

---------------